In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold
import joblib
from sklearn.metrics import classification_report

# 1. Load and Preprocess Data
def load_and_preprocess():
    """Load and prepare the training data"""
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Feature engineering
    data['date'] = pd.to_datetime(data['date'])
    data['rest_to_eat_ratio'] = np.where(
        (data['EAT'] == 0) & (data['REST'] == 0),
        1.0,
        data['REST'] / (data['EAT'] + 1e-6)
    )
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'rest_to_eat_ratio']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    return data[features], data[targets], data['cow']

# 2. Train and Save Model
def train_model():
    """Train and save the multi-output Random Forest model"""
    X, y, groups = load_and_preprocess()
    
    # Create base Random Forest classifier with balanced class weights
    base_rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1  # Use all available cores
    )
    
    # Wrap in MultiOutputClassifier
    model = MultiOutputClassifier(base_rf)
    
    # Cross-validated training
    gkf = GroupKFold(n_splits=3)
    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Print evaluation metrics
        print("\nClassification Report:")
        for i, target in enumerate(y.columns):
            print(f"\n{target}:")
            print(classification_report(y_test.iloc[:, i], y_pred[:, i]))
    
    # Save the final model
    joblib.dump(model, 'cattle_disease_model_rf.joblib')
    print("\nRandom Forest model trained and saved successfully.")

# 3. Prediction Function
def predict_diseases(new_data):
    """
    Make predictions on new cattle data using Random Forest
    Args:
        new_data: DataFrame containing:
            - IN_ALLEYS
            - REST
            - EAT
            - ACTIVITY_LEVEL
            - date (YYYY-MM-DD format)
    Returns:
        DataFrame with predictions and probabilities for each disease
    """
    # Load model
    model = joblib.load('cattle_disease_model_rf.joblib')
    
    # Preprocess new data (same as training)
    new_data = new_data.copy()
    new_data['date'] = pd.to_datetime(new_data['date'])
    new_data['rest_to_eat_ratio'] = np.where(
        (new_data['EAT'] == 0) & (new_data['REST'] == 0),
        1.0,
        new_data['REST'] / (new_data['EAT'] + 1e-6))
    
    # Prepare features
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'rest_to_eat_ratio']
    X_new = new_data[features]
    
    # Make predictions
    predictions = model.predict(X_new)
    probabilities = model.predict_proba(X_new)  # Returns list of arrays
    
    # Format results
    results = pd.DataFrame({
        'oestrus': predictions[:, 0],
        'oestrus_prob': probabilities[0][:, 1],
        'calving': predictions[:, 1],
        'calving_prob': probabilities[1][:, 1],
        'lameness': predictions[:, 2],
        'lameness_prob': probabilities[2][:, 1],
        'mastitis': predictions[:, 3],
        'mastitis_prob': probabilities[3][:, 1]
    })
    
    return results

# 4. Example Usage
if __name__ == "__main__":
    # First train the model
    train_model()
    
    # Example new data for prediction
    new_data = pd.DataFrame({
        'IN_ALLEYS': [35.6, 0.0, 10.2],
        'REST': [3564.4, 3599.9, 2500.0],
        'EAT': [0.0, 0.0, 500.0],
        'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
        'date': ['2023-01-01', '2023-01-02', '2023-01-03']
    })
    
    # Get predictions
    predictions = predict_diseases(new_data)
    print("\nDisease Predictions:")
    print(predictions)


Classification Report:

oestrus:
              precision    recall  f1-score   support

           0       1.00      0.87      0.93    642649
           1       0.01      0.33      0.02      2544

    accuracy                           0.87    645193
   macro avg       0.50      0.60      0.48    645193
weighted avg       0.99      0.87      0.93    645193


calving:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96    643777
           1       0.01      0.57      0.03      1416

    accuracy                           0.92    645193
   macro avg       0.51      0.74      0.49    645193
weighted avg       1.00      0.92      0.95    645193


lameness:
              precision    recall  f1-score   support

           0       1.00      0.73      0.84    644089
           1       0.00      0.33      0.00      1104

    accuracy                           0.73    645193
   macro avg       0.50      0.53      0.42    645193
weighted avg       